# geo0647 – DWD Climate Stations: from the Station List to a CSV (Demo)

**Unit:** geo0640 DWD CDC Stations & Time Series

The German Weather Service (DWD) publishes the list of its climate stations as a plain text file.
This notebook turns that file into two clean CSV tables that QGIS can show as point layers:

1. **all** stations of the annual climate data (KL),
2. the stations in **North Rhine-Westphalia (NRW) that are still active**.

```
opendata.dwd.de  →  download by hand  →  look at the text  →  column names  →  pandas DataFrame  →  check  →  filter  →  2 CSV files  →  QGIS
```

**This is a demo.** You do not have to write code: run the cells one after the other (**Shift + Enter**) and read the
explanations. Every step is explained in class. Writing such a reader yourself is a separate activity (geo0646).

**Result:** `data/derived/dwd_kl_stations.csv` and `data/derived/dwd_kl_stations_nrw_active.csv`

## 1. Download the station file by hand
1. Open the DWD directory of the annual climate data in your browser:
   <https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/annual/kl/historical/>
   It is a plain list of files. Look around: one zip file per station, plus a few description files.
2. Find **`KL_Jahreswerte_Beschreibung_Stationen.txt`** (*Jahreswerte* = annual values, *Beschreibung* = description,
   *Stationen* = stations). Right-click it → **Save link as…** and save it on your laptop.
   Do **not** copy the text from the browser window into an editor: that changes the encoding of the file (see §2).
3. Run the next cell. It creates the folder `data/original/dwd/annual/kl/historical/` next to this notebook.
   The folder names follow the path on the DWD server, so you can always tell where a file came from.
4. In the JupyterLab file browser (left), double-click your way into that folder and click **Upload** (arrow icon on top).
   Choose the file you saved. Keep the file name exactly as it is.
5. Run the cell after next. It checks that the file is in the right place.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Folders: original downloads are never edited, everything we produce goes to data/derived
ORIG_DIR = Path("data/original/dwd/annual/kl/historical")   # mirrors the path on opendata.dwd.de
OUT_DIR = Path("data/derived")
ORIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

STATION_FILE = ORIG_DIR / "KL_Jahreswerte_Beschreibung_Stationen.txt"
print("pandas", pd.__version__)
print("Upload the station file into:", ORIG_DIR.resolve())

In [ ]:
# Is the file there?
if not STATION_FILE.exists():
    print("Files in", ORIG_DIR, ":", [p.name for p in ORIG_DIR.iterdir()])
    raise FileNotFoundError(f"{STATION_FILE} not found. Upload it into this folder (step 4) and keep its name.")
print("ok:", STATION_FILE, STATION_FILE.stat().st_size, "bytes")

## 2. Look at the file before you read it
**Predict:** Is there a separator like `,` or `;`? How are dates written? Which special characters do you expect in German station names?

The file is not UTF-8 but **Latin-1** (ISO 8859-1), an older encoding for Western European characters.
We read all lines as text and look at the first ones. `repr()` also shows invisible characters such as `\n` (end of line) and trailing blanks.

In [ ]:
with open(STATION_FILE, encoding="latin-1") as f:
    lines = f.readlines()

print(len(lines), "lines")
for line in lines[:4]:
    print(repr(line[:130]))

# Check: a file saved from an editor as UTF-8 reads without error, but 'ü' becomes 'Ã¼'
assert not any("Ã" in line for line in lines), "The file is not Latin-1 any more. Download it again with 'Save link as…'."
print("ok: encoding looks right (Baden-Württemberg)")

What we see:
* **Line 1:** the column names, separated by blanks.
* **Line 2:** a row of dashes (decoration, no data).
* **From line 3:** the data in **fixed-width columns**: every value starts at the same position in each line,
  the gaps are filled with blanks. There is no separator, and station names may contain blanks themselves.
* **Dates** are written as `YYYYMMDD`, e.g. `19310101` = 1 January 1931.
* **Station ids** have leading zeros: `00001`, `05792`. The zeros belong to the id, so we keep it as **text**, not as a number.

## 3. Column names: from the header line, translated with a dictionary
The header line is one string. `split()` cuts it at every run of blanks, `strip()` removes blanks and line ends at both ends.

The names are German. We translate them with a **dictionary** (German name → English name). The original file stays unchanged:
the translation is part of the code, so it is documented and repeatable.

| German (in the file) | meaning | English name |
|---|---|---|
| `Stations_id` | station identifier | `station_id` |
| `von_datum` | date from (first measurement) | `date_from` |
| `bis_datum` | date to (last measurement) | `date_to` |
| `Stationshoehe` | station altitude (*Höhe*) in m | `altitude` |
| `geoBreite` | geographic latitude (*Breite*) φ, y | `lat` |
| `geoLaenge` | geographic longitude (*Länge*) λ, x | `lon` |
| `Stationsname` | station name | `name` |
| `Bundesland` | federal state | `state` |
| `Abgabe` | release of the data (*Frei* = freely available) | `access` |

In [ ]:
header = lines[0]
names_de = [s.strip() for s in header.split()]
names_de

In [ ]:
COLUMNS_DE_EN = {
    "Stations_id": "station_id",
    "von_datum": "date_from",
    "bis_datum": "date_to",
    "Stationshoehe": "altitude",
    "geoBreite": "lat",
    "geoLaenge": "lon",
    "Stationsname": "name",
    "Bundesland": "state",
    "Abgabe": "access",
}

# Every name of the header must be in the dictionary; if DWD changes the file, we notice it here
missing = [n for n in names_de if n not in COLUMNS_DE_EN]
assert not missing, f"not in the dictionary: {missing}"

names = [COLUMNS_DE_EN[n] for n in names_de]   # English names, in the order of the file
names

## 4. Read the data with `pd.read_fwf`
`read_fwf` reads **f**ixed-**w**idth **f**ormatted text. We tell it:
* `skiprows=2`: skip the header and the dashes, we give the names ourselves (`names=`),
* `encoding="latin-1"`,
* `dtype={"station_id": str}`: the id stays text with its leading zeros,
* `parse_dates=[...]`, `date_format="%Y%m%d"`: turn `19310101` into a date.

pandas **guesses** where each column starts and ends by looking at the data. By default it looks at the **first 100 lines only**.

**Predict:** the highest DWD station is on the **Zugspitze**, Germany's highest mountain (2 962 m). What will the table say?

In [ ]:
read_options = dict(skiprows=2, names=names, encoding="latin-1",
                    dtype={"station_id": str},
                    parse_dates=["date_from", "date_to"], date_format="%Y%m%d")

df_guess = pd.read_fwf(STATION_FILE, **read_options)                         # column borders from the first 100 lines
df = pd.read_fwf(STATION_FILE, **read_options, infer_nrows=len(lines))        # column borders from all lines

print("first 100 lines:", df_guess.loc[df_guess["station_id"] == "05792", "altitude"].item(), "m")
print("all lines:      ", df.loc[df["station_id"] == "05792", "altitude"].item(), "m")
print("raw line:       ", repr([line for line in lines if line.startswith("05792")][0][:60]))
print((df_guess["altitude"] != df["altitude"]).sum(), "altitudes and",
      (df_guess["name"] != df["name"]).sum(), "names damaged by the guess from 100 lines")

In the first 100 lines no altitude has 4 digits, so pandas makes the altitude column too narrow and the `2` of `2956` is cut off,
**without any error message**. With `infer_nrows=len(lines)` pandas looks at all lines (the file is short) and gets it right.
From now on we only use `df`.

> A reader that does not crash is not necessarily a reader that is right. Check the result against something you know.

The **data types** of the columns (compare with the attribute types of a vector layer):

In [ ]:
df.dtypes

## 5. Quick checks
How many stations? Missing values? Plausible values? Germany lies roughly between φ = 47° and 55.1° and λ = +5.8° and +15.1°.

In [ ]:
print(len(df), "stations,", df["station_id"].is_unique and "ids unique")
print(df.isna().sum().to_string())
df[["altitude", "lat", "lon", "date_from", "date_to"]].describe()

In [ ]:
# The five highest stations
df.sort_values("altitude", ascending=False).head(5)[["station_id", "name", "altitude", "state"]]

The empty values in `access` are not a problem for us: the release status is simply missing for a few stations.

## 6. Filter: stations in NRW that are still active
* **In NRW:** `state` is `"Nordrhein-Westfalen"` (the file keeps the German names of the federal states).
* **Still active:** the station delivered data up to the most recent end date in the whole file.

**Predict:** NRW has about 90 stations in the file. How many are still active?

In [ ]:
latest = df["date_to"].max()
print("most recent end date in the file:", latest.date())

in_nrw = df["state"] == "Nordrhein-Westfalen"
active = df["date_to"] == latest
df_nrw_active = df[in_nrw & active]

print(in_nrw.sum(), "stations in NRW,", len(df_nrw_active), "of them still active")
df_nrw_active[["station_id", "name", "altitude", "date_from"]]

## 7. Save as CSV
Choices for the two files:
* **separator `;`**: station names contain commas (`Bergzabern, Bad`),
* **encoding UTF-8**: the standard today, QGIS detects it,
* **dates as `YYYY-MM-DD`** (ISO 8601): sortable as text, recognised by QGIS as dates,
* **no index column** (`index=False`): `station_id` is already a column,
* coordinates unchanged: `lon` (x) and `lat` (y) in degrees, WGS 84 = **EPSG:4326**.

Next to each CSV we write a small **`.csvt`** file with one line of field types. GDAL (the library QGIS uses to read files)
then knows that `station_id` is text and does not turn `05792` into the number 5792.

In [ ]:
CSVT_TYPES = {"station_id": "String", "date_from": "Date", "date_to": "Date", "altitude": "Integer",
              "lat": "Real", "lon": "Real", "name": "String", "state": "String", "access": "String"}


def write_csv(table, path):
    # Write table as ;-separated UTF-8 CSV plus a .csvt file with the field types
    table.to_csv(path, sep=";", encoding="utf-8", index=False, date_format="%Y-%m-%d")
    path.with_suffix(".csvt").write_text(",".join(f'"{CSVT_TYPES[c]}"' for c in table.columns) + "\n")
    print(f"{path}: {len(table)} stations, {path.stat().st_size:,} bytes (+ {path.with_suffix('.csvt').name})")


csv_all = OUT_DIR / "dwd_kl_stations.csv"
csv_nrw = OUT_DIR / "dwd_kl_stations_nrw_active.csv"
write_csv(df, csv_all)
write_csv(df_nrw_active, csv_nrw)

**Check** the result as text (this is what QGIS will see) and by reading it back:

In [ ]:
with open(csv_all, encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline().rstrip())
print(csv_all.with_suffix(".csvt").read_text())

check = pd.read_csv(csv_all, sep=";", dtype={"station_id": str}, parse_dates=["date_from", "date_to"])
zugspitze = check[check["station_id"] == "05792"].iloc[0]
assert len(check) == len(df) and zugspitze["altitude"] == 2956
print("ok:", len(check), "stations,", zugspitze["name"], zugspitze["station_id"], zugspitze["altitude"], "m")

## 8. In QGIS
1. In JupyterLab, right-click each file in `data/derived/` → **Download**: both `.csv` files and their `.csvt` files. Keep them in one folder.
2. QGIS: *Layer → Add Layer → Add Delimited Text Layer…*
   * **File format:** Custom delimiters, tick **Semicolon**,
   * **Record and Fields Options:** *First record has field names*, *Detect field types*,
   * **Geometry definition:** *Point coordinates*, **X field `lon`**, **Y field `lat`**, **Geometry CRS: EPSG:4326 – WGS 84**,
   * in the sample table at the bottom, check that `station_id` is **Text**; change it if not.
3. **Check:** does *Aachen* (`00003`) sit on Aachen? Is the Zugspitze (`05792`) on the Austrian border?

## 9. Wrap-up
* **Look** at a file before you read it: separator, encoding, header lines, date format.
* **Take the names from the data** (header line) and translate them with a **dictionary** in code, never by editing the original file.
* **Ids are text**: `05792` keeps its zeros.
* **Check the result** against something you know (the Zugspitze): a silent error is worse than a crash.
* **Filter** with conditions on columns, combined with `&`.

**Homework and presentations:** see the course plan in the README of your course repository.